# Sanity Check: Remapped Grounding Boxes on Combined Images

Plots `viz_grounding_clinical_remapped` and `viz_grounding_dscope_remapped`
boxes overlaid on combined images. No GPU needed — reads pre-computed
results from the CSV.

In [ ]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

PROJECT_ROOT = "/scratch/jq2uw/derm_vlms"
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
IMAGES_DIR = os.path.join(RESULTS_DIR, "images")

In [ ]:
MODEL = "dermato_llama"  # change to "gpt53" or "medgemma"

df = pd.read_csv(os.path.join(RESULTS_DIR, f"{MODEL}_predictions_reason_viz.csv"))
combined = df[df["image_mode"] == "combined"].reset_index(drop=True)
print(f"Model: {MODEL}  |  Combined rows: {len(combined)}")
combined[["id", "y16", "ground_truth"]].head()

In [ ]:
COLORS = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45",
    "#fabed4", "#469990", "#dcbeff", "#9A6324",
]


def draw_boxes(ax, image, entries, label_prefix="", linestyle="-"):
    """Draw grounding boxes on an axes, colored by sentence index."""
    W, H = image.size
    for i, entry in enumerate(entries):
        box = entry.get("box")
        if not box:
            continue
        color = COLORS[i % len(COLORS)]
        rect = patches.Rectangle(
            (box["x"] * W, box["y"] * H),
            box["w"] * W, box["h"] * H,
            linewidth=2, edgecolor=color, facecolor="none",
            linestyle=linestyle,
        )
        ax.add_patch(rect)
        is_diag = entry.get("type") == "diagnosis"
        prefix = f"[D] " if is_diag else ""
        tag = f"{label_prefix}{prefix}{entry.get('text', '')[:30]}"
        ax.text(
            box["x"] * W, box["y"] * H - 4,
            tag, color="white", fontsize=6.5,
            bbox=dict(facecolor=color, alpha=0.75, pad=1.5),
        )


def plot_remapped(row, figsize=(18, 8)):
    """Plot a combined image with clinical (solid) and dscope (dashed) boxes."""
    img_path = os.path.join(IMAGES_DIR, f"{row['id']}.jpg")
    image = Image.open(img_path).convert("RGB")

    clin = json.loads(str(row.get("viz_grounding_clinical_remapped", "[]")))
    dscope = json.loads(str(row.get("viz_grounding_dscope_remapped", "[]")))

    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(image)

    draw_boxes(ax, image, clin, label_prefix="", linestyle="-")
    draw_boxes(ax, image, dscope, label_prefix="", linestyle="--")

    # Draw split line
    num = row["id"].replace("_combined", "")
    p = Image.open(os.path.join(IMAGES_DIR, f"{num}_photo.jpg"))
    d = Image.open(os.path.join(IMAGES_DIR, f"{num}_dscope.jpg"))
    h = min(p.height, d.height)
    pw = int(p.width * h / p.height)
    dw = int(d.width * h / d.height)
    split_x = pw / (pw + dw) * image.width
    ax.axvline(x=split_x, color="white", linewidth=1.5, linestyle=":")
    ax.text(split_x - 60, 30, "clinical", color="white", fontsize=9,
            bbox=dict(facecolor="black", alpha=0.5, pad=2))
    ax.text(split_x + 10, 30, "dscope", color="white", fontsize=9,
            bbox=dict(facecolor="black", alpha=0.5, pad=2))

    ax.set_title(
        f"{row['id']}  |  GT: {row['ground_truth']}  |  y16: {row['y16']}\n"
        f"solid = clinical box, dashed = dscope box  |  colors = per sentence",
        fontsize=10,
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()


print(f"Plotting functions ready.")

## Plot a few examples

In [ ]:
N_EXAMPLES = 5

for i in range(min(N_EXAMPLES, len(combined))):
    row = combined.iloc[i]
    plot_remapped(row)

## Pick a specific case by ID

In [ ]:
CASE_ID = "6_combined"  # change to any combined ID

row = combined[combined["id"] == CASE_ID].iloc[0]
plot_remapped(row)

## Side-by-side: original separate images vs remapped on combined

Shows the raw clinical/dscope boxes on their own images (left, middle)
alongside the remapped boxes on the combined image (right).

In [ ]:
def plot_side_by_side(row):
    """3-panel: clinical-only, dscope-only, combined with remapped boxes."""
    num = row["id"].replace("_combined", "")
    img_clin = Image.open(os.path.join(IMAGES_DIR, f"{num}_photo.jpg")).convert("RGB")
    img_dscope = Image.open(os.path.join(IMAGES_DIR, f"{num}_dscope.jpg")).convert("RGB")
    img_combined = Image.open(os.path.join(IMAGES_DIR, f"{num}_combined.jpg")).convert("RGB")

    clin_raw = json.loads(str(row.get("viz_grounding_clinical", "[]")))
    dscope_raw = json.loads(str(row.get("viz_grounding_dscope", "[]")))
    clin_remap = json.loads(str(row.get("viz_grounding_clinical_remapped", "[]")))
    dscope_remap = json.loads(str(row.get("viz_grounding_dscope_remapped", "[]")))

    fig, axes = plt.subplots(1, 3, figsize=(24, 8))

    # Left: clinical only with raw boxes
    axes[0].imshow(img_clin)
    draw_boxes(axes[0], img_clin, clin_raw, linestyle="-")
    axes[0].set_title("Clinical (raw boxes)", fontsize=10)
    axes[0].axis("off")

    # Middle: dscope only with raw boxes
    axes[1].imshow(img_dscope)
    draw_boxes(axes[1], img_dscope, dscope_raw, linestyle="--")
    axes[1].set_title("Dscope (raw boxes)", fontsize=10)
    axes[1].axis("off")

    # Right: combined with remapped boxes
    axes[2].imshow(img_combined)
    draw_boxes(axes[2], img_combined, clin_remap, linestyle="-")
    draw_boxes(axes[2], img_combined, dscope_remap, linestyle="--")
    axes[2].set_title("Combined (remapped: solid=clinical, dashed=dscope)", fontsize=10)
    axes[2].axis("off")

    fig.suptitle(
        f"{row['id']}  |  GT: {row['ground_truth']}  |  y16: {row['y16']}",
        fontsize=12, y=1.01,
    )
    plt.tight_layout()
    plt.show()


for i in range(min(3, len(combined))):
    plot_side_by_side(combined.iloc[i])